[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/practicas/Practica_5.ipynb)

# Práctica 5: Determinantes de la satisfacción del cliente
## Enfoque en interpretación de determinantes

## Objetivo de aprendizaje
Construir e interpretar un modelo de regresión lineal múltiple, en particular:
- La selección y justificación de las variables independientes
- La R cuadrada y la R cuadrada ajustada como medidas de ajuste
- La significancia global del modelo y la de cada coeficiente
- La interpretación del signo y la magnitud de los coeficientes
- La evaluación de los supuestos del modelo

## Estructura del ejercicio
Esta práctica tiene una alternativa, la Práctica 5b (estimación del ingreso del hogar). Ambas cubren el mismo contenido estadístico, por lo que **debes elegir solo una de las dos**.

En la sección de "Contexto y preparación de datos" únicamente debes
- Sustituir la industria por una diferente a la del ejemplo (1001, alimentos procesados)

En la sección de "Análisis e interpretación":
- Es donde debes decidir qué variables entran al modelo y escribir tu código
- Es donde debes realizar la interpretación estadística y de negocio del modelo

A diferencia de las prácticas anteriores, aquí **no se te indica qué variables usar**. Elegirlas y defender esa elección es la parte central de la actividad.

## Contexto y preparación de datos
Supongamos que trabajas en el área de investigación de mercados y necesitas saber **qué determina la satisfacción de los clientes** en una industria específica. La respuesta orienta decisiones concretas: si la satisfacción depende sobre todo de la calidad percibida, la inversión debe ir a la operación y al servicio; si depende del precio dado la calidad, el problema es de posicionamiento y de propuesta de valor.

El archivo `acsi2015` contiene una muestra de la encuesta con la que se construye el índice de satisfacción del cliente estadounidense (ACSI: American Customer Satisfaction Index). La encuesta original recopila información de más de 400,000 consumidores pertenecientes a más de 400 empresas en aproximadamente 50 industrias en Estados Unidos.

La muestra proporcionada incluye información de 8,239 consumidores pertenecientes a las siguientes industrias:

- 1001 = "Processed Food (Nondurables)"
- 3003 = "Commercial Airlines (Transportation)"
- 3013 = "Internet Service Providers (Telecommunications)"
- 5001 = "Commercial Banks (Finance)"

El archivo está localizado en: https://github.com/adan-rs/amd/raw/main/data/acsi2015.xlsx

La descripción de las variables es la siguiente:

|VARIABLE	|LABEL	|VALUE|
|-------------|--------|----------|
|INDUSTRY	|Industry Code	| Industry Code|
|YEAR	|Year in which data collected	|Year|
|SATIS	|Overall Customer Satisfaction	|1="Very dissatisfied"; 10="Very satisfied"|
|CONFIRM	|Confirmation to Expectations	|1="Falls short of expectations"; 10="Exceeds expectations"|
|IDEAL	|Close to ideal product/service	|1="Not very close to ideal"; 10="Very close to ideal"|
|OVERALLX	|Expectation about overall quality	|1="Not very high"; 10="Very high"|
|CUSTOMX	|Expectations about customization	|1="Not very high"; 10="Very high"|
|WRONGX	|Expectation about reliability	|1="Not very high"; 10="Very high"|
|OVERALLQ	|Overall Quality	|1="Not very good"; 10="Very good"|
|CUSTOMQ	|Meeting personal requirement (Customization)	|1="Not very good"; 10="Very good"|
|WRONGQ	|Things went wrong (Reliability)	|1="Not very good"; 10="Very good"|
|PQ	|Price given Quality	|1="Not very good price given quality"; 10="Very good price given quality"|
|QP	|Quality given Price	|1="Not very good quality given price"; 10="Very good quality given price"|
|COMP	|Customer complaints	|1="Yes"; 0="No"|
|REPUR	|Repurchase Intention	|1="Not very likely"; 10="Very likely"|
|AGE	|Age	|Age|
|EDUCAT	|Education	|1=Less than high school; 2=High school; 3=Some college or associate degree; 4=College graduate; 5=Post-graduate|
|HISPANIC	|Hispanic	|1="Yes"; 0="No"|
|RACE_1	|Race_1	|1="White"; 2="Black/African-American"; 3="American Indian/Alaskan Native"; 4="Asian"; 5="Native Hawaiian or Pacific Islander"; 6="Other Race"|
|INCOME	|Income	|1="Under 20K"; 2="20K to 30K"; 3="30K to 40K"; 4="40K to 60K"; 5="60K to 80K"; 6="80K to 100K"; 7="100K or More"|
|GENDER	|Gender	|1="Male"; 2="Female"|
|ZIPCODE	|Zip code	|Respondent zip code|

> **Advertencia: el modelo requiere tu criterio, no el de la computadora.** Que una variable esté en la base de datos no significa que pueda entrar al modelo. Antes de estimar, revisa una por una las variables disponibles y decide si tiene sentido usarlas como predictoras de `SATIS`. Al menos tres situaciones distintas exigen esa revisión:
>
> - **Variables que miden lo mismo que la dependiente.** `IDEAL` es uno de los tres reactivos con los que el ACSI construye el índice de satisfacción: no es una causa de la satisfacción, es otra forma de preguntarla. Incluirla produce una R cuadrada alta y un modelo que no explica nada, porque estarías explicando la satisfacción con la satisfacción. **No la incluyas como variable independiente.** En la base hay al menos otra variable con exactamente este problema: identifícala, exclúyela y explica en tu análisis por qué.
> - **Variables que son consecuencia y no causa.** Alguna de las variables disponibles ocurre *después* de la satisfacción y es resultado de ella. Un modelo que la use como predictora invierte la dirección del razonamiento. Detéctala y decide qué hacer, justificando la decisión.
> - **Variables cuyo número no es una cantidad.** Hay códigos e identificadores que son etiquetas, no magnitudes, y que no deben entrar como variables numéricas. Otras son categóricas u ordinales y, si decides usarlas, requieren un tratamiento explícito (variables indicadoras, o justificar por qué las tratas como continuas).
>
> Un modelo con la R cuadrada más alta no es el mejor modelo si sus variables no resisten estas tres preguntas. Se evaluará la justificación de lo que **dejaste fuera** con el mismo peso que lo que incluiste.

> **Nota: alcance didáctico del ejercicio.** El modelo que estimarás aquí es una versión simplificada con fines de aprendizaje. Antes de interpretar los resultados, ten presente que:
> - El archivo es una **muestra preparada para el curso**, no la base completa del ACSI, y no incluye los ponderadores del diseño muestral. Los resultados no son estimaciones oficiales del índice ni son representativos de la población de consumidores.
> - Los datos corresponden a **consumidores estadounidenses en 2015**. Las conclusiones no se trasladan automáticamente a otro país ni a la situación actual de la industria.
> - Se trata de un **corte transversal**: se observa a cada consumidor una sola vez, por lo que el modelo no describe cómo cambia la satisfacción en el tiempo.
> - Las variables de percepción se midieron en una escala de 1 a 10 y aquí se tratan como continuas. Es un supuesto de conveniencia, razonable con diez niveles, pero es un supuesto.
> - Todas las respuestas provienen del **mismo cuestionario y del mismo informante**, lo que tiende a inflar las correlaciones entre percepciones (varianza de método común).
> - El modelo es descriptivo: mide **asociación, no efecto causal**. Nada fue asignado aleatoriamente entre los consumidores.
> - El ACSI real no se estima con una regresión múltiple, sino con un modelo de ecuaciones estructurales con variables latentes. Aquí usamos la regresión por su valor didáctico.
>
> Estas limitaciones no invalidan el ejercicio, pero sí acotan las conclusiones que puedes defender.

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

La celda siguiente descarga el archivo y filtra la industria seleccionada. **Utiliza una industria diferente a la del ejemplo (1001)**: elige entre 3003, 3013 o 5001, y justifica tu elección en el análisis.

In [ ]:
# Parámetros
industria = 1001    # Cambiar por otra industria: 3003, 3013 o 5001

# Importar el archivo "data/acsi2015.xlsx"
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/acsi2015.xlsx')

# Filtrar la industria seleccionada
datos = df[df['INDUSTRY'] == industria].copy()

print(f'Observaciones totales:  {len(df)}')
print(f'Observaciones en la industria {industria}: {len(datos)}')

In [ ]:
# Revisar las variables y el número de observaciones
datos.info()

Antes de decidir el modelo conviene ver cuántos datos faltantes tiene cada variable en la industria elegida. Una variable con muchos faltantes no es inservible, pero si la incluyes, la regresión se estimará solo con los casos completos y eso reduce la muestra.

In [ ]:
# Datos faltantes por variable en la industria seleccionada
faltantes = pd.DataFrame({
    'faltantes': datos.isna().sum(),
    '% faltantes': (datos.isna().mean() * 100).round(1)
})
faltantes.sort_values('% faltantes', ascending=False)

## Análisis e interpretación
Construye un modelo para explicar la satisfacción de los clientes (`SATIS`) en función de las variables disponibles en la base de datos, para la industria que seleccionaste.

Tu análisis debe incluir, como mínimo, los siguientes apartados:

**1. Planteamiento del modelo**
- Especifica la variable dependiente.
- Define las variables independientes seleccionadas.

**2. Justificación del modelo**
- Justifica teóricamente o analíticamente la elección de las variables independientes.
- Explica por qué dichas variables podrían influir en la satisfacción del cliente.
- Justifica también las **exclusiones**: qué variables descartaste y por qué, siguiendo la advertencia de la sección anterior.

**3. Preparación de los datos**
- De ser necesario, haz un manejo apropiado de datos perdidos o atípicos, y documenta el criterio utilizado.
- Si usas variables categóricas, explica cómo las codificaste.

**4. Tamaño muestral**
- Verifica que el tamaño de la muestra sea suficiente, considerando el número de variables explicativas utilizadas.

**5. Estadística descriptiva**
- Presenta la estadística descriptiva de las variables cuantitativas utilizadas en el modelo.
- Interpreta brevemente los resultados obtenidos.

**6. Estimación del modelo**
- Ajusta un modelo de regresión lineal múltiple.
- Presenta la ecuación estimada del modelo.

**7. Evaluación del ajuste del modelo**
- Reporta e interpreta el coeficiente de determinación (R² y R² ajustada).
- Explica por qué al comparar modelos con distinto número de variables conviene mirar la R² ajustada y no la R².

**8. Evaluación de la significancia del modelo**
- Interpreta el p-valor del estadístico F.
- Concluye si el modelo es globalmente significativo.

**9. Evaluación de la significancia de los coeficientes**
- Interpreta los p-valores de los coeficientes.
- Identifica qué variables son estadísticamente significativas.

**10. Interpretación de resultados**
- Interpreta el signo y magnitud de los coeficientes.
- Explica el impacto de las variables significativas sobre la satisfacción del cliente.
- Deriva al menos una implicación para la industria analizada.

**11. Evaluación de supuestos del modelo**

Analiza al menos un posible problema relacionado con:
- Linealidad
- Normalidad de los residuos
- Homocedasticidad
- Multicolinealidad
- Independencia de los errores

## Uso de IA generativa (para extender, no para resolver)

La IA se usa para **ampliar** la práctica partiendo de lo que ya construiste, no para hacerla. Declara la herramienta y la versión utilizada (por ejemplo, ChatGPT 5, Claude Opus 4.5, Gemini 3 Pro, Copilot).

**Qué debes entregar** (cuatro bloques, en celdas de texto dentro del notebook):

1. **Tu pregunta de negocio.** Una pregunta propia, pertinente y que la práctica **no** responda. Formúlala en primera persona: "Quiero saber si...", "Me preocupa que...". No se acepta reproducir la pregunta del ejercicio ni preguntas genéricas. Del tipo esperado (no para copiar): *"Quiero saber si el peso del precio dado la calidad es distinto entre los clientes que se quejaron y los que no, porque eso cambiaría a quién dirigir la recuperación del servicio"*.
2. **El prompt completo, transcrito en celda de texto** (no de código). Debe incluir: rol, objetivo, tu código, **los resultados reales de tu regresión pegados** (la salida de `summary()`) y una restricción explícita de lo obvio (por ejemplo: "no me expliques qué es la R cuadrada ni qué es la multicolinealidad"). Un prompt de una línea, sin contexto ni resultados, no cuenta.
3. **Qué adopté y qué descarté.** De la respuesta recibida, indica qué implementaste y qué dejaste fuera, con la razón. La implementación usa **máximo 50 líneas de código**.
4. **Verificación.** Un cálculo, contraejemplo o comprobación que valide o refute algo que la IA afirmó. Por ejemplo: si la IA propone agregar una variable que mejora la R cuadrada, verifica que esa variable no sea otra medición de la satisfacción; si afirma que hay multicolinealidad severa, calcula los factores de inflación de la varianza y contrasta.

**Evidencia auditable**: pega el resultado completo de la extensión (texto, tablas o código), no un enlace a la conversación. Revisa que el documento o el código no quede cortado.

**No se acepta**:
- Transferir las instrucciones de la práctica a la IA, ya sea copiándolas o parafraseándolas, para que ella la resuelva.
- Usar la IA como enciclopedia: respuestas generales sin tus datos ni tu código. Está bien usarla para comprender, pero debe haber contenido propio nuevo.
- Transcribir sugerencias sin implementarlas, o dar por cierto lo que la IA afirma sin comprobarlo.
- Delegar la interpretación: las conclusiones y su redacción son tuyas.
- Aceptar una selección automática de variables (por ejemplo, un procedimiento paso a paso sugerido por la IA) sin revisar si las variables elegidas tienen sentido.

**Buenas prácticas sugeridas**: repreguntar a la IA sobre su propia respuesta; pedirle explícitamente las limitaciones de lo que propone; traer un concepto externo al curso y aplicarlo a tus datos.

**Defensa oral**: cualquier práctica puede ser seleccionada al azar para una defensa oral breve (3 a 5 minutos), en la que deberás explicar tus decisiones, tu código y tus conclusiones. Un trabajo que no pueda ser explicado por su autor se considerará evidencia de trabajo no auténtico y podrá ser penalizado.

## Entregable
Notebook en Jupyter exportado a pdf o html, con el código, análisis e interpretación.

## Rúbrica de evaluación
1. **Planteamiento y justificación del modelo (20%)**: se evalúan los puntos 1 y 2. Incluye la justificación de las variables seleccionadas y, con el mismo peso, la de las **excluidas**. Se anula este criterio si el modelo usa como predictora una variable que mide la satisfacción misma (por ejemplo `IDEAL`).
2. **Preparación de los datos y tamaño muestral (10%)**: se evalúan los puntos 3 y 4. Tratamiento documentado de datos faltantes y atípicos, codificación explícita de variables categóricas y verificación de que la muestra sostiene el número de predictores.
3. **Análisis descriptivo (10%)**: se evalúa el punto 5. Se penalizará presentar tablas sin interpretación.
4. **Estimación y ajuste del modelo (15%)**: se evalúan los puntos 6, 7 y 8. Modelo correctamente estimado, ecuación presentada, e interpretación de R², R² ajustada y estadístico F. Se penalizará reportar los números sin interpretarlos.
5. **Interpretación de los coeficientes (15%)**: se evalúan los puntos 9 y 10. Interpretación del signo, la magnitud y la significancia de cada coeficiente, con al menos una implicación para la industria analizada. Se penalizará que sólo se indique si el coeficiente es significativo o no.
6. **Evaluación de supuestos (10%)**: se evalúa el punto 11. Diagnóstico realizado con evidencia (gráficos o pruebas) y lectura de lo que implica para la validez del modelo.
7. **Extensión del análisis con IA generativa (20%)**, evaluada en cuatro partes iguales (5% cada una):
   - *Pregunta propia (5%)*: pregunta de negocio formulada por el alumno, pertinente y no respondida por la práctica. Se anula si reproduce la pregunta del ejercicio o si es genérica.
   - *Calidad del prompt (5%)*: transcrito completo en celda de texto, con rol, objetivo, código y los resultados reales de la regresión, más una restricción explícita de lo obvio. Se anula si se transfieren las instrucciones de la práctica (copiadas o parafraseadas).
   - *Ejecución: qué adopté y qué descarté (5%)*: lo propuesto se implementa (máximo 50 líneas de código) y se explica qué quedó fuera y por qué. No basta transcribir sugerencias.
   - *Verificación e interpretación propia (5%)*: un cálculo, contraejemplo o comprobación que valide o refute una afirmación de la IA, y conclusiones redactadas por el alumno.

   *Requisito de forma*: la evidencia debe ser auditable, esto es, resultado completo pegado en el notebook, sin cortes y sin enlaces a la conversación como único respaldo.

La claridad de la redacción y el orden del notebook se consideran de manera transversal: respuestas vagas, confusas o no alineadas con los resultados obtenidos serán penalizadas en el criterio correspondiente.

Se permite el uso de herramientas de apoyo (incluyendo IA) como soporte técnico y, como se indica en la sección anterior, para profundizar en el análisis. Sin embargo, el análisis, la interpretación de resultados y las conclusiones deben reflejar comprensión propia.

Respuestas genéricas, excesivamente uniformes o que no estén alineadas con los resultados obtenidos en el notebook podrán considerarse como evidencia de trabajo no auténtico y serán evaluadas con penalización.